# 04 — Splits user-level sobre corpus multi-fuente (codigo inline)
**Etapa:** 2b - Multi-corpus EN->ES

Genera splits 70/10/20 (train/val/test) **estratificados por user_id** sobre
el corpus unificado. Critico: el split es por `user_id`, no por documento,
para evitar que el mismo usuario aparezca en train y test (leakage).

**Toda la logica de split esta inline** - no usa `src.data.build_splits`.

**Pre-requisito**: `03_merge_multi.ipynb` ejecutado (`data/processed/corpus_v1.parquet`).

**Output**: `data/processed/splits_multi/{train,val,test}.parquet` +
`split_manifest.json` con breakdown por fuente.

In [ ]:
# --- Imports y setup ---
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

PROCESSED = Path("data/processed")
SEED = 42
TRAIN_P, VAL_P, TEST_P = 0.7, 0.1, 0.2

assert (PROCESSED / "corpus_v1.parquet").exists(), \
    "Falta corpus_v1.parquet - ejecuta 03_merge_multi.ipynb primero"
np.random.seed(SEED)
print(f"Repo: {REPO_ROOT}")
print("Corpus OK")

In [ ]:
# --- Split user-level estratificado (inline) ---
def assign_users(user_df, train, val, test, seed):
    """Asigna cada user_id a {train, val, test} estratificando por label dominante."""
    modal = (
        user_df.groupby("user_id")["label"]
        .agg(lambda s: int(s.mode().iloc[0]) if not s.mode().empty else int(s.iloc[0]))
        .reset_index()
        .rename(columns={"label": "modal_label"})
    )
    train_val, test_part = train_test_split(
        modal, test_size=test, stratify=modal["modal_label"], random_state=seed
    )
    val_ratio = val / (train + val)
    train_part, val_part = train_test_split(
        train_val, test_size=val_ratio, stratify=train_val["modal_label"], random_state=seed
    )
    out = {}
    for uid in train_part["user_id"]:
        out[uid] = "train"
    for uid in val_part["user_id"]:
        out[uid] = "val"
    for uid in test_part["user_id"]:
        out[uid] = "test"
    return out


df = pd.read_parquet(PROCESSED / "corpus_v1.parquet")
print(f"corpus: {len(df):,} filas, {df['user_id'].nunique():,} usuarios")

user_assignment = assign_users(df, TRAIN_P, VAL_P, TEST_P, SEED)
df["split"] = df["user_id"].map(user_assignment)
assert df["split"].notna().all(), "quedaron usuarios sin split"
print(f"\nSplit assignment:\n{df['split'].value_counts()}")

In [ ]:
# --- Escribir train/val/test.parquet + manifest ---
out_dir = PROCESSED / "splits_multi"
out_dir.mkdir(parents=True, exist_ok=True)
manifest = {
    "seed": SEED,
    "fractions": {"train": TRAIN_P, "val": VAL_P, "test": TEST_P},
}
for split in ("train", "val", "test"):
    sub = df[df["split"] == split]
    out_path = out_dir / f"{split}.parquet"
    sub.drop(columns=["split"]).to_parquet(out_path, index=False)
    manifest[split] = {
        "n_rows": int(len(sub)),
        "n_users": int(sub["user_id"].nunique()),
        "by_label": {str(k): int(v) for k, v in sub["label"].value_counts().items()},
        "by_corpus_source": {str(k): int(v) for k, v in sub["corpus_source"].value_counts().items()},
    }
    print(f"  -> {out_path}: {len(sub):,} filas, {sub['user_id'].nunique():,} usuarios")

(out_dir / "split_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False)
)
print("-> split_manifest.json")

In [ ]:
# --- Verificar manifest: val/test deben tener representacion multi-fuente ---
manifest = json.load(open("data/processed/splits_multi/split_manifest.json", encoding="utf-8"))
print(json.dumps(manifest, indent=2, ensure_ascii=False))

for split in ["train", "val", "test"]:
    df_split = pd.read_parquet(
        f"data/processed/splits_multi/{split}.parquet",
        columns=["label", "corpus_source"],
    )
    print(f"\n{split}: {len(df_split):,} filas | {df_split['corpus_source'].nunique()} fuentes")
    print(f"  fuentes: {df_split['corpus_source'].value_counts().to_dict()}")
    print(f"  labels:  {df_split['label'].value_counts().to_dict()}")

for split in ["val", "test"]:
    df_split = pd.read_parquet(
        f"data/processed/splits_multi/{split}.parquet", columns=["corpus_source"]
    )
    assert df_split["corpus_source"].nunique() >= 2, \
        f"{split} tiene solo {df_split['corpus_source'].nunique()} fuente - el split es degenerado"
print("\nOK - val/test tienen representacion multi-fuente")

## Conclusiones

- Splits user-level generados en `data/processed/splits_multi/`:
  - `train.parquet`, `val.parquet`, `test.parquet`
  - `split_manifest.json` con breakdown por fuente y label
- Val y test tienen representacion multi-fuente (validacion cross-corpus).
- **Siguiente**: re-entrenar BETO con `notebooks/04_beto/03_finetune_beto_multi.ipynb`.